# Implementing Byte Pair Encoding

In [1]:
pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


## Implementation of Dataloader

The Dataset stores and provides individual input-target pairs, while the DataLoader groups those pairs into batches for training.

In [1]:
# Install pytorch
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):

    # max_length : Number of tokens in each input sequence
    # stride : Number of positions to move the sliding window
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        # Sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunks = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunks))



    def __len__(self):
        return len(self.input_ids)



    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]



def create_dataloader_v1(txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):
    
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training.
        num_workers=num_workers # The number of CPU processes to use for preprocessing
    )

    return dataloader


with open("../../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

    dataloader = create_dataloader_v1(
        raw_text,
        batch_size=1,
        max_length=4,
        stride=1,
        shuffle=False
    )

    data_iter = iter(dataloader)
    first_batch = next(data_iter)
    print(first_batch)

    second_batch = next(data_iter)
    print(second_batch)



c:\Users\rites\Desktop\LLM_from_scratch\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


## Creating the embeddings

In [6]:
vocab_size = 50257
output_dim = 256
# PyTorch creates a trainable weight matrix with  50257 rows and 256 columns.
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Instantiate the data loader
max_len = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_len, stride=max_len, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)


# convert tokenIds to embeddings

# Embedding layer 1 : Token Embedding
token_embedding = token_embedding_layer(inputs)
print(token_embedding.shape)

# Embedding layer 2 : positional embedding
contex_length = max_len
pos_embedding_layer = torch.nn.Embedding(contex_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(contex_length))
print(pos_embeddings.shape)

# final input embedding
input_embedding = token_embedding + pos_embeddings
print(input_embedding.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])
torch.Size([8, 4, 256])
torch.Size([4, 256])
torch.Size([8, 4, 256])
